# 99 Full Pipeline - Complete Analysis Workflow

This notebook demonstrates:
- Running the complete analysis pipeline from scratch
- Checkpoint management for resume capability
- Config export for reproducibility
- Excel report generation
- Final results validation

**⚠️ Warning**: This notebook runs the full pipeline and may take 5-15 minutes depending on data size.

## Setup

In [ ]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from datetime import datetime
import time

# Import project modules
from src._01_setup import config_loader, checkpoint_manager, config_exporter
from src._02_preprocessing import data_cleaner
from src._03_clustering.pipeline import ClusteringPipeline

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)

print("✓ Imports complete")
print(f"📅 Analysis started: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

## 1. Load and Configure

Load default configuration and make any final adjustments.

In [ ]:
# Load configuration
config = config_loader.load()

print("✓ Configuration loaded")
print("\n📋 Current Settings:")
print(f"  • Market: {config.get('market', 'germany')}")
print(f"  • K-Means clusters: {config['clustering']['kmeans']['n_clusters']}")
print(f"  • Hierarchical clusters: {config['clustering']['hierarchical']['n_clusters']}")
print(f"  • DBSCAN eps: {config['clustering']['dbscan']['eps']}")

# Make final adjustments here if needed
# Example:
# config['clustering']['kmeans']['n_clusters'] = 7

print("\n💡 Edit configuration above if needed, then proceed")

## 2. Initialize Checkpoint Manager

In [ ]:
# Setup checkpoint manager
manager = checkpoint_manager.CheckpointManager(market=config.get('market', 'germany'))

print("✓ Checkpoint manager initialized")
print(f"  Checkpoint directory: {manager.checkpoint_dir}")

# List existing checkpoints
existing_checkpoints = manager.list_checkpoints()
if existing_checkpoints:
    print(f"\n📦 Found {len(existing_checkpoints)} existing checkpoint type(s):")
    for name, timestamps in existing_checkpoints.items():
        print(f"  • {name}: {len(timestamps)} version(s)")
else:
    print("\n📦 No existing checkpoints found")

## 3. Run Full Pipeline

Execute the complete analysis pipeline with all algorithms.

In [ ]:
print("🚀 Starting full pipeline...\n")
start_time = time.time()

try:
    # Initialize pipeline
    pipeline = ClusteringPipeline(config)
    
    # Run pipeline
    print("Running clustering pipeline (this may take several minutes)...")
    results = pipeline.run()
    
    elapsed_time = time.time() - start_time
    
    print(f"\n✓ Pipeline completed successfully!")
    print(f"⏱️  Total time: {elapsed_time/60:.1f} minutes")
    
    # Save checkpoint
    print("\n💾 Saving checkpoint...")
    checkpoint_file = manager.save_checkpoint(
        results,
        name='full_pipeline_complete',
        metadata={
            'timestamp': datetime.now().isoformat(),
            'runtime_minutes': elapsed_time / 60,
            'config_snapshot': {
                'kmeans_k': config['clustering']['kmeans']['n_clusters'],
                'hierarchical_k': config['clustering']['hierarchical']['n_clusters'],
                'dbscan_eps': config['clustering']['dbscan']['eps']
            }
        }
    )
    print(f"✓ Checkpoint saved: {Path(checkpoint_file).name}")
    
except Exception as e:
    print(f"\n❌ Pipeline failed with error:")
    print(f"   {str(e)}")
    print(f"\n⚠️  Check error details above and restart if needed")
    raise

## 4. Validate Results

Quick validation of pipeline outputs.

In [ ]:
if 'results' in locals() and results:
    print("📊 Pipeline Results Summary:\n")
    
    # Check what's in results
    print(f"Result keys: {list(results.keys())}")
    
    # Load and check consolidated data
    output_path = Path(f"../output/{config.get('market', 'germany')}/01_data")
    
    if (output_path / 'assignments.csv').exists():
        df_assignments = pd.read_csv(output_path / 'assignments.csv')
        print(f"\n✓ Assignments: {len(df_assignments)} companies")
        print(f"  • Clusters: {df_assignments['cluster'].nunique()}")
        print(f"  • Cluster names: {df_assignments['cluster_name'].unique().tolist()}")
    
    if (output_path / 'profiles.csv').exists():
        df_profiles = pd.read_csv(output_path / 'profiles.csv')
        print(f"\n✓ Profiles: {len(df_profiles)} cluster profiles")
    
    # Check algorithm outputs
    algo_path = Path(f"../output/{config.get('market', 'germany')}/02_algorithms")
    algorithms = ['kmeans_comparative', 'hierarchical', 'dbscan']
    
    print(f"\n✓ Algorithm Results:")
    for algo in algorithms:
        if (algo_path / algo).exists():
            print(f"  • {algo}: ✓")
        else:
            print(f"  • {algo}: ⚠ Not found")
    
    # Check Excel reports
    excel_path = Path(f"../output/{config.get('market', 'germany')}/04_excel_reports")
    if excel_path.exists():
        excel_files = list(excel_path.glob('**/*.xlsx'))
        print(f"\n✓ Excel Reports: {len(excel_files)} file(s) generated")
        for f in excel_files:
            size_mb = f.stat().st_size / 1024 / 1024
            print(f"  • {f.name}: {size_mb:.1f} MB")
else:
    print("⚠ No results to validate - run pipeline first")

## 5. Export Configuration for Reproducibility

In [ ]:
# Export configuration snapshot
exporter = config_exporter.ConfigExporter(config, market=config.get('market', 'germany'))

# Export YAML
config_file = exporter.export_config(
    format='yaml'
)
print(f"✓ Config exported (YAML): {config_file}")

# Export run summary
if 'elapsed_time' in locals():
    summary_file = exporter.export_run_summary(
        runtime_minutes=elapsed_time / 60,
        n_companies=len(df_assignments) if 'df_assignments' in locals() else None,
        n_clusters=df_assignments['cluster'].nunique() if 'df_assignments' in locals() else None,
        algorithms_run=['kmeans', 'hierarchical', 'dbscan']
    )
    print(f"✓ Run summary exported: {summary_file}")

print("\n💾 All configuration saved for reproducibility")

## 6. Quick Visualization of Results

In [ ]:
if 'df_assignments' in locals() and 'df_profiles' in locals():
    # Cluster sizes
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    
    cluster_sizes = df_assignments['cluster'].value_counts().sort_index()
    cluster_sizes.plot(kind='bar', ax=ax1, color='steelblue')
    ax1.set_title('Cluster Sizes', fontsize=14, fontweight='bold')
    ax1.set_xlabel('Cluster')
    ax1.set_ylabel('Number of Companies')
    ax1.tick_params(axis='x', rotation=0)
    
    # Cluster profiles heatmap
    features = ['roa', 'roe', 'ebit_margin', 'gross_margin', 'fcf_margin']
    profile_data = df_profiles[features].T
    
    sns.heatmap(profile_data, annot=True, fmt='.2f', cmap='RdYlGn', 
                center=0, ax=ax2, cbar_kws={'label': 'Value'})
    ax2.set_title('Cluster Profiles (Key Metrics)', fontsize=14, fontweight='bold')
    ax2.set_xlabel('Cluster')
    ax2.set_ylabel('Metric')
    
    plt.suptitle('Pipeline Results Overview', fontsize=16, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.show()
    
    print("\n✓ Visualization complete")
else:
    print("⚠ No data available for visualization")

## 7. Final Summary

In [ ]:
print("="*80)
print("PIPELINE EXECUTION COMPLETE")
print("="*80)

print(f"\n📅 Completed: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

if 'elapsed_time' in locals():
    print(f"⏱️  Total runtime: {elapsed_time/60:.1f} minutes")

if 'df_assignments' in locals():
    print(f"\n📊 Results:")
    print(f"  • Companies analyzed: {len(df_assignments)}")
    print(f"  • Clusters identified: {df_assignments['cluster'].nunique()}")
    print(f"  • Checkpoints saved: {len(manager.list_checkpoints())}")

print(f"\n📂 Output locations:")
print(f"  • Data: output/{config.get('market', 'germany')}/01_data/")
print(f"  • Algorithms: output/{config.get('market', 'germany')}/02_algorithms/")
print(f"  • Excel: output/{config.get('market', 'germany')}/04_excel_reports/")
print(f"  • Config: output/{config.get('market', 'germany')}/00_config/")

print(f"\n💡 Next steps:")
print(f"  1. Review Excel reports in 04_excel_reports/")
print(f"  2. Use 01_quick_start.ipynb for quick data exploration")
print(f"  3. Use 03_cluster_exploration.ipynb for detailed cluster analysis")
print(f"  4. Use 04_algorithm_comparison.ipynb for robustness validation")

print("\n" + "="*80)
print("✓ Ready for thesis analysis!")
print("="*80)

## Recovery from Checkpoint

If pipeline was interrupted, you can resume from the last checkpoint:

In [ ]:
# Uncomment to load from checkpoint instead of running full pipeline:

# manager = checkpoint_manager.CheckpointManager(market='germany')
# results = manager.load_checkpoint('full_pipeline_complete')

# if results:
#     print("✓ Checkpoint loaded successfully")
#     print(f"  Keys: {list(results.keys())}")
#     # Continue analysis from here
# else:
#     print("⚠ No checkpoint found - run full pipeline first")

## Notes for Thesis

### Configuration Used
- Exported to: `output/germany/00_config/analysis_config.yaml`
- Run summary: `output/germany/00_config/run_summary.yaml`

### Key Files for Thesis
1. **research_analysis_master.xlsx**: Main results structured by research questions
2. **algorithm_comparison_combined.xlsx**: Robustness validation
3. **company_cluster_analysis_*.xlsx**: Detailed cluster-level analysis

### Reproducibility
To reproduce these exact results:
1. Use the config file in `output/germany/00_config/analysis_config.yaml`
2. Run this notebook with that configuration
3. All random seeds and parameters are documented

### Citation
```
Analysis performed on [date]
Configuration: output/germany/00_config/analysis_config.yaml
Runtime: [X] minutes
```